<a href="https://colab.research.google.com/github/MananAslamDev/ML-Stuff/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (Grain): One row represents one unique content item (web page) for a specific client on a specific day.
Time Window: To train a model, we need a "feature window" (e.g., looking at 90 days of past performance) to predict a "target window" (e.g., the subsequent 30 days). For this data contract and verification, we are examining a single mid-panel month: March 2026.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features (Observable Signals): gsc_impressions, clicks, position, content_age_days, ga4_data_available. These are safe to use because they are historical facts knowable before any decision needs to be made.

Label (Proxy): trend_direction == 'down'. This is the outcome proxy we want to predict (whether a page is declining).

Context: client_hash_id, content_hash_id, report_date. These are used for joining and grouping, not as features.

Excluded: trend_pct (leaks the label, as the label is mathematically derived from it). We also strictly exclude any product-level scores like health_score or action_type. Why: Including product scores creates a circular model that just learns to memorize the company's existing if/else rules instead of finding true search patterns.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
import duckdb
import pandas as pd
from google.colab import userdata

# 1. Connect to Hugging Face using your secret token
hf_token = userdata.get('HF_TOKEN')
conn = duckdb.connect()
conn.execute(f"""
    INSTALL httpfs;
    LOAD httpfs;
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# The correct path to the partitioned dataset on Hugging Face
base_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

# Query 1: Verify the Grain (One row = one day x client x content)
query_1 = f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as duplicate_count
    FROM '{base_url}/month=2026-03/*.parquet'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
"""
duplicates = conn.execute(query_1).df()
print("--- Query 1: Grain Verification ---")
print(duplicates)
print("Conclusion: Grain is verified. The dataframe is empty, meaning there are no duplicate rows for a single day/content/client combination.\n")

# Query 2: Counts and Missing Values (Availability)
query_2 = f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(gsc_impressions) as rows_with_impressions,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4
    FROM '{base_url}/month=2026-03/*.parquet'
"""
counts = conn.execute(query_2).df()
print("--- Query 2: Counts and Availability ---")
print(counts)
print(f"Percentage of rows with GA4 data available: {(counts['rows_with_ga4'][0] / counts['total_rows'][0]) * 100:.1f}%\n")

# Query 3: Verify the Window
query_3 = f"""
    SELECT
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM '{base_url}/month=2026-03/*.parquet'
"""
window = conn.execute(query_3).df()
print("--- Query 3: Window Verification ---")
print(window)
print("Conclusion: The dates strictly bound to the expected month of March 2026.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Query 1: Grain Verification ---
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, duplicate_count]
Index: []
Conclusion: Grain is verified. The dataframe is empty, meaning there are no duplicate rows for a single day/content/client combination.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Query 2: Counts and Availability ---
   total_rows  rows_with_impressions  rows_with_ga4
0     9841378                9841378       413966.0
Percentage of rows with GA4 data available: 4.2%

--- Query 3: Window Verification ---
  start_date   end_date
0 2026-03-01 2026-03-31
Conclusion: The dates strictly bound to the expected month of March 2026.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced History: Not all clients have data going back equally far. If a client was onboarded in January 2026, they won't have data in 2025. This means we cannot expect a perfect 365-day history for every page.

GSC-only Early Rows: Before a client connects their GA4 account, ga4_data_available is FALSE. This means sessions will be 0 or null. We have to be careful not to mistake "tracking wasn't set up yet" for "this page got zero traffic."

Window Overlaps: When creating rolling features (e.g., 90-day averages), we must strictly ensure the feature window closes before the target window begins, or we risk time-leakage.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.